In [1]:
import random
from scipy.stats import ttest_ind_from_stats
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import fisher_exact
import math
import re

# Data Processing

## Get the A/B test data for headlines

In [2]:
#Combine three subsets
#70% upworthy-archive-confirmatory-packages-03.12.2020 
#15% upworthy-archive-exploratory-packages-03.12.2020
#15% upworthy-archive-holdout-packages-03.12.2020
csv_file_path_1 = 'upworthy-archive-datasets/upworthy-archive-exploratory-packages-03.12.2020.csv'
csv_file_path_2 = 'upworthy-archive-datasets/upworthy-archive-confirmatory-packages-03.12.2020.csv'
csv_file_path_3 = 'upworthy-archive-datasets/upworthy-archive-holdout-packages-03.12.2020.csv'
df1 = pd.read_csv(csv_file_path_1)
df2 = pd.read_csv(csv_file_path_2)
df3 = pd.read_csv(csv_file_path_3)

df = pd.concat([df1, df2, df3])
df.to_csv('upworthy-archive-datasets/upworthy-archive-packages-all.csv', index=False)

/opt/anaconda3/envs/env_full/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (15) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
def filter_groups(group):
    return group if len(group['headline'].unique()) > 1 else None

def find_headline_pairs_with_numbers(group):
    pairs = []
    sorted_group = group.sort_values('CTR', ascending=False).reset_index()
    for i in range(len(sorted_group) - 1):
        for j in range(i + 1, len(sorted_group)):
            headline_1 = sorted_group.iloc[i]
            headline_2 = sorted_group.iloc[j]
            # Compare the CTR values and assign 1 if the first headline's CTR is higher, else 2
            higher_ctr = 0
            if headline_1['CTR'] > headline_2['CTR']:
                higher_ctr = 1
            else:
                higher_ctr = 2
            clicks = [headline_1['clicks'], headline_2['clicks']]
            non_clicks = [headline_1['impressions'] -  headline_1['clicks'], headline_2['impressions'] -  headline_2['clicks']]
            contingency_table = [clicks, non_clicks]
            #chi2, p, dof, expected = chi2_contingency(contingency_table)
            odds_ratio, p_value = fisher_exact(contingency_table)
            if headline_1['headline'] != headline_2['headline']:
                pairs.append({
                    'clickability_test_id': headline_1['clickability_test_id'],
                    'eyecatcher_id': headline_1['eyecatcher_id'],
                    'new_test_id': headline_1['new_test_id'],
                    'headline_1': headline_1['headline'],
                    'headline_2': headline_2['headline'],
                    'higher_CTR': higher_ctr,
                    'CTR_1': headline_1['CTR'],
                    'CTR_2': headline_2['CTR'],
                    'Size_1': headline_1['impressions'],
                    'Size_2': headline_2['impressions'],
                    'p_value': p_value
                })
    return pd.DataFrame(pairs)

def shuffle_and_exchange(group):
    # Shuffle the group
    shuffled_group = group.sample(frac=1).reset_index(drop=True)
    
    # Exchange the headlines with 50% probability
    for i in shuffled_group.index:
        if np.random.rand() < 0.5:
            # Swap the headlines
            shuffled_group.at[i, 'headline_1'], shuffled_group.at[i, 'headline_2'] = \
                shuffled_group.at[i, 'headline_2'], shuffled_group.at[i, 'headline_1']
            shuffled_group.at[i, 'CTR_1'], shuffled_group.at[i, 'CTR_2'] = \
                shuffled_group.at[i, 'CTR_2'], shuffled_group.at[i, 'CTR_1']
            shuffled_group.at[i, 'Size_1'], shuffled_group.at[i, 'Size_2'] = \
                shuffled_group.at[i, 'Size_2'], shuffled_group.at[i, 'Size_1']
            shuffled_group.at[i, 'higher_CTR'] = 2 if shuffled_group.at[i, 'higher_CTR'] == 1 else 1
    
    return shuffled_group


In [5]:
df['clicks'] = df['clicks'].astype(float) 
df['impressions'] = df['impressions'].astype(float) 
df['CTR'] = df['clicks']/df['impressions']
filtered_groups = df.groupby(['clickability_test_id', 'eyecatcher_id']).apply(filter_groups).reset_index(drop=True)
new_df = filtered_groups[['clickability_test_id', 'eyecatcher_id', 'headline', 'CTR','clicks', 'impressions']].drop_duplicates()
new_df.to_csv('upworthy-archive-datasets/ctr-all.csv', index=False)
del df, new_df

## Generate pairs of headlines

In [9]:
df = pd.read_csv('upworthy-archive-datasets/ctr-all.csv')
df = df[['clickability_test_id', 'eyecatcher_id', 'headline','impressions','clicks']]
#assign new test id because in the original same clickability_test_id, eyecatcher can be different
df['new_test_id'] = df.groupby(['clickability_test_id', 'eyecatcher_id']).ngroup() + 1

num_unique_new_test_id = df['new_test_id'].nunique()
print(f"Number of tests: {num_unique_new_test_id}")
print(f"Number of packages: {len(df)}")
print(f"Number of impressions: {df['impressions'].sum()}")
print(f"Number of clicks: {df['clicks'].sum()}")
df = df[['new_test_id, 'headline','impressions','clicks']]

df['clicks'] = df['clicks'].astype(float) 
df['impressions'] = df['impressions'].astype(float) 
df['CTR'] = df['clicks']/df['impressions']

headline_pairs_with_numbers_df = df.groupby(['new_test_id']).apply(find_headline_pairs_with_numbers).reset_index(drop=True)
# shuffle the data to make it balanced, which means the random guess only achieve maximum 0.5 accuracy
headline_pairs_with_numbers_df = headline_pairs_with_numbers_df.groupby(['new_test_id'], group_keys=False).apply(shuffle_and_exchange)

print('------after combining three datasets------')
print('# of original data samples = ', len(df))
print('------after choosing tests in headlines------')
print('# of tested headlines = ', len(df))
print('------after getting pairs------')
print('# of headline pairs = ', len(headline_pairs_with_numbers_df))
print('------among which, (fisher_exact test)------')
print('# of 90% significant pairs = ', len(headline_pairs_with_numbers_df[headline_pairs_with_numbers_df['p_value']<0.1]))
print('# of 95% significant pairs = ', len(headline_pairs_with_numbers_df[headline_pairs_with_numbers_df['p_value']<0.05]))
print('# of 99% significant pairs = ', len(headline_pairs_with_numbers_df[headline_pairs_with_numbers_df['p_value']<0.01]))


headline_pairs_with_numbers_df.to_csv('upworthy-archive-datasets/winner-all-new.csv', index=False)

Number of tests': 17681
Number of packages': 77245
Number of impressions': 277338713.0
Number of clicks': 3741517.0
------after combining three datasets------
# of original data samples =  77245
------after choosing tests in headlines------
# of tested headlines =  77245
------after getting pairs------
# of headline pairs =  140655
------among which, (fisher_exact test)------
# of 90% significant pairs =  49867
# of 95% significant pairs =  39158
# of 99% significant pairs =  23682


In [10]:
headline_pairs_with_numbers_df.head(10)

,new_test_id,headline_1,headline_2,higher_CTR,CTR_1,CTR_2,Size_1,Size_2,p_value
0,1,If You Know Anyone Who Is Afraid Of Gay People...,I've Got Some News For You. Being Gay Is Genet...,1,0.028881,0.009615,4155.0,4160.0,7.622063e-11
1,1,If You Know Anyone Who Is Afraid Of Gay People...,"Hey Dude. If You Have An Older Brother, There'...",1,0.028881,0.010049,4155.0,4080.0,4.156051e-10
2,1,"Here's The Science, Here's The Gay. Open Your ...",I've Got Some News For You. Being Gay Is Genet...,1,0.013271,0.009615,4069.0,4160.0,1.210542e-01
3,1,"Hey Dude. If You Have An Older Brother, There'...",I've Got Some News For You. Being Gay Is Genet...,1,0.010049,0.009615,4080.0,4160.0,9.112003e-01
4,1,"SCIENCE FACT: Gay Science, Like Straight Scien...","Hey Dude. If You Have An Older Brother, There'...",2,0.007744,0.010049,4132.0,4080.0,2.909000e-01
5,1,"Here's The Science, Here's The Gay. Open Your ...",If You Know Anyone Who Is Afraid Of Gay People...,2,0.013271,0.028881,4069.0,4155.0,7.409125e-07
6,1,"Here's The Science, Here's The Gay. Open Your ...","Hey Dude. If You Have An Older Brother, There'...",1,0.013271,0.010049,4069.0,4080.0,1.811862e-01
7,1,"SCIENCE FACT: Gay Science, Like Straight Scien...",I've Got Some News For You. Being Gay Is Genet...,2,0.007744,0.009615,4132.0,4160.0,4.077244e-01
8,1,"SCIENCE FACT: Gay Science, Like Straight Scien...",If You Know Anyone Who Is Afraid Of Gay People...,2,0.007744,0.028881,4132.0,4155.0,2.265775e-13
9,1,"SCIENCE FACT: Gay Science, Like Straight Scien...","Here's The Science, Here's The Gay. Open Your ...",2,0.007744,0.013271,4132.0,4069.0,1.666367e-02
